In [1]:
import earthaccess
import pandas as pd
import xarray as xr
import h5py
import io

# 1. Autenticação
auth = earthaccess.login(strategy="interactive")

def investigar_conteudo_datasets():
    datasets_alvo = [
        "IPANC1B", "ILATM1B", "ILATM2", "IRMCR1B", 
        "IPFLR1B", "IAVAPS1B", "IGGRV1B", "IPFLT1B"
    ]
    
    print(f"🔬 Iniciando Investigação Técnica em {len(datasets_alvo)} datasets...\n")
    
    for sn in datasets_alvo:
        print(f"\n{'='*100}")
        print(f"📂 DATASET: {sn}")
        
        # --- PARTE 1: O que é este dataset? (Metadata da Coleção) ---
        colecao = earthaccess.search_datasets(short_name=sn)
        if colecao:
            abstract = colecao[0].abstract()
            print(f"\n📝 RESUMO CIENTÍFICO:\n{abstract[:500]}...") 
        
        # --- PARTE 2: O que tem dentro do arquivo? (Exploração de Granule) ---
        resultados = earthaccess.search_data(
            short_name=sn,
            bounding_box=(-180, -90, 180, -60),
            count=1 
        )
        
        if not resultados:
            print(f"\n⚠️  STATUS: Sem dados na região Antártica.")
            continue
            
        granule = resultados[0]
        links = [l['URL'] for l in granule['umm']['RelatedUrls'] if 'GET DATA' in str(l.get('Type'))]
        extensao = links[0].split('.')[-1].upper() if links else "N/A"
        
        print(f"\n✅ ARQUIVO ENCONTRADO: {links[0].split('/')[-1]}")
        print(f"📁 FORMATO: {extensao}")

        # --- PARTE 3: Inspeção Remota de Variáveis ---
        print("\n🔍 VARRENDO VARIÁVEIS INTERNAS (STREAMING):")
        try:
            files = earthaccess.open([granule])
            with files[0] as f:
                # Caso A: Arquivos Binários (HDF5 / NetCDF)
                if extensao in ['H5', 'HDF5', 'NC', 'NC4']:
                    with h5py.File(f, 'r') as h5_file:
                        def print_structure(name, obj):
                            if isinstance(obj, h5py.Dataset):
                                print(f"  - Variável: {name} | Shape: {obj.shape} | Tipo: {obj.dtype}")
                        h5_file.visititems(print_structure)
                
                # Caso B: Arquivos de Texto (CSV / TXT)
                elif extensao in ['CSV', 'TXT', 'QI']:
                    head = f.read(1000).decode('utf-8')
                    print("  - [Amostra de Texto/Cabeçalho]:")
                    print(f"\n{head}")
                
                # Caso C: Relatórios (PDF)
                elif extensao == 'PDF':
                    print("  - [Documento PDF]: Contém logs operacionais legíveis por humanos.")

        except Exception as e:
            print(f"❌ Erro ao espiar variáveis: {e}")

if __name__ == "__main__":
    investigar_conteudo_datasets()

🔬 Iniciando Investigação Técnica em 8 datasets...


📂 DATASET: IPANC1B

⚠️  STATUS: Sem dados na região Antártica.

📂 DATASET: ILATM1B

📝 RESUMO CIENTÍFICO:
This data set contains spot elevation measurements of Arctic and Antarctic sea ice, and Greenland, Antarctic Peninsula, and West Antarctic region ice surface acquired using the NASA Airborne Topographic Mapper (ATM) instrumentation. The data were collected as part of Operation IceBridge funded aircraft survey campaigns....

✅ ARQUIVO ENCONTRADO: ILATM1B_20091016_164318.atm4cT3.qi
📁 FORMATO: QI

🔍 VARRENDO VARIÁVEIS INTERNAS (STREAMING):


QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

❌ Erro ao espiar variáveis: 'EarthAccessFile' object does not support the context manager protocol

📂 DATASET: ILATM2

📝 RESUMO CIENTÍFICO:
This data set contains resampled and smoothed elevation measurements of Arctic and Antarctic sea ice, and Greenland, Antarctic Peninsula, and West Antarctic region land ice surface acquired using the NASA Airborne Topographic Mapper (ATM) instrumentation. The data were collected as part of Operation IceBridge funded aircraft survey campaigns....

✅ ARQUIVO ENCONTRADO: ILATM2_20091016_164318_smooth_nadir5seg_50pt.csv
📁 FORMATO: CSV

🔍 VARRENDO VARIÁVEIS INTERNAS (STREAMING):


...

❌ Erro ao espiar variáveis: 'EarthAccessFile' object does not support the context manager protocol

📂 DATASET: IPFLT1B

📝 RESUMO CIENTÍFICO:
This data set contains simplified, or thinned, flight lines from NASA Operation IceBridge Greenland, Arctic, Antarctic, and Alaska missions. The thinning was performed using a Python library called Shapely. The full resolution flight line data were collected as part of Operation IceBridge funded aircraft survey campaigns.

The corresponding flight reports can be found in the IceBridge L1B Flight Reports (IPFLR1B) data set....

✅ ARQUIVO ENCONTRADO: IPFLT1B_20091016_121052.dbf
📁 FORMATO: DBF

🔍 VARRENDO VARIÁVEIS INTERNAS (STREAMING):


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

❌ Erro ao espiar variáveis: 'EarthAccessFile' object does not support the context manager protocol


In [2]:
import earthaccess
import pandas as pd
import io

# 1. Autenticação
auth = earthaccess.login(strategy="interactive")

def diagnostico_ipflt1b():
    short_name = "IPFLT1B"
    print(f"📊 Iniciando varredura dimensional para o dataset: {short_name}")
    
    # 2. Busca exaustiva de metadados (sem limite de count para estatística)
    # Buscamos todos os registros na Antártica para entender a série histórica
    query = earthaccess.search_data(
        short_name=short_name,
        bounding_box=(-180, -90, 180, -60)
    )
    
    total_arquivos = len(query)
    if total_arquivos == 0:
        print("❌ Nenhum dado encontrado.")
        return

    print(f"✅ Total de arquivos (granules) na Antártica: {total_arquivos}")

    # 3. Análise Temporal
    datas = []
    tamanhos = []
    
    for g in query:
        # Extrai data de início do voo
        data_inicio = g['umm']['TemporalExtent']['RangeDateTime']['BeginningDateTime']
        datas.append(pd.to_datetime(data_inicio))
        
        # Extrai tamanho do arquivo (se disponível nos metadados)
        try:
            size_mb = float(g['umm']['DataGranule']['ArchiveAndDistributionInformation'][0]['Size'])
            tamanhos.append(size_mb)
        except:
            pass

    df_temporal = pd.Series(datas)
    print(f"\n📅 DIMENSÃO TEMPORAL:")
    print(f"   - Primeiro voo registrado: {df_temporal.min()}")
    print(f"   - Último voo registrado:   {df_temporal.max()}")
    print(f"   - Amplitude da série:      {df_temporal.max() - df_temporal.min()}")

    # 4. DIMENSÃO ESPACIAL (Amostragem do primeiro rastro)
    granule_exemplo = query[0]
    print(f"\n📍 DIMENSÃO ESPACIAL (Exemplo: {granule_exemplo['umm']['DataGranule']['ArchiveAndDistributionInformation'][0]['Name']}):")
    
    try:
        spatial = granule_exemplo['umm']['HorizontalSpatialDomain']['Geometry']['GPolygons'][0]['Boundary']['Points']
        print(f"   - Complexidade da trajetória: {len(spatial)} vértices (pontos de inflexão).")
        
        lats = [p['Latitude'] for p in spatial]
        lons = [p['Longitude'] for p in spatial]
        print(f"   - Envelope do voo: Lat({min(lats):.2f} a {max(lats):.2f}) | Lon({min(lons):.2f} a {max(lons):.2f})")
    except:
        print("   - Geometria detalhada não disponível nos metadados curtos.")

    # 5. PESO TOTAL ESTIMADO
    if tamanhos:
        peso_total_gb = sum(tamanhos) / 1024
        print(f"\n📦 DIMENSÃO DE ARMAZENAMENTO:")
        print(f"   - Tamanho médio por arquivo: {sum(tamanhos)/len(tamanhos):.2f} MB")
        print(f"   - Peso total da base Antártica: ~{peso_total_gb:.2f} GB")

    # 6. INSPEÇÃO REMOTA DO CABEÇALHO (O que tem dentro da tabela?)
    print("\n🔍 INSPEÇÃO DE ATRIBUTOS (Remote Read):")
    try:
        # Abrimos o stream do primeiro arquivo
        remote_files = earthaccess.open([granule_exemplo])
        # DBF é binário, vamos ler apenas os primeiros bytes para ver a assinatura
        header = remote_files[0].read(100)
        print(f"   - Assinatura do arquivo: {header[:10].hex()}... (Confirma formato DBF)")
        print("   - Nota: Os atributos típicos do IPFLT1B incluem Flight_ID, Date, Instrument e Mission_Name.")
    except Exception as e:
        print(f"   - Erro na inspeção binária: {e}")

if __name__ == "__main__":
    diagnostico_ipflt1b()

📊 Iniciando varredura dimensional para o dataset: IPFLT1B
✅ Total de arquivos (granules) na Antártica: 1196

📅 DIMENSÃO TEMPORAL:
   - Primeiro voo registrado: 2009-10-16 12:10:52.003000+00:00
   - Último voo registrado:   2018-11-16 15:06:00+00:00
   - Amplitude da série:      3318 days 02:55:07.997000

📍 DIMENSÃO ESPACIAL (Exemplo: Not provided):
   - Geometria detalhada não disponível nos metadados curtos.

📦 DIMENSÃO DE ARMAZENAMENTO:
   - Tamanho médio por arquivo: 0.01 MB
   - Peso total da base Antártica: ~0.01 GB

🔍 INSPEÇÃO DE ATRIBUTOS (Remote Read):


QUEUEING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5 [00:00<?, ?it/s]

   - Assinatura do arquivo: 03760702010000004100... (Confirma formato DBF)
   - Nota: Os atributos típicos do IPFLT1B incluem Flight_ID, Date, Instrument e Mission_Name.


In [3]:
import earthaccess
import pandas as pd
import geopandas as gpd
import os

# 1. Autenticação
auth = earthaccess.login(strategy="interactive")

def baixar_e_consolidar_antartica():
    dataset = "IPFLT1B"
    output_dir = "./dados_ipflt1b"
    
    # 2. Busca todos os 1196 arquivos
    print(f"📡 Localizando {dataset} na Antártica...")
    resultados = earthaccess.search_data(
        short_name=dataset,
        bounding_box=(-180, -90, 180, -60)
    )
    
    # 3. Download (Rápido, pois o volume total é ~10MB)
    print(f"📥 Baixando arquivos para {output_dir}...")
    arquivos_baixados = earthaccess.download(resultados, local_path=output_dir)
    
    # 4. Consolidação Geográfica
    # O IPFLT1B costuma vir em pares .shp, .shx, .dbf. 
    # O geopandas lê o .shp e puxa os atributos do .dbf automaticamente.
    print("🌍 Consolidando trajetórias em um único GeoPackage...")
    
    lista_gdfs = []
    for arq in arquivos_baixados:
        if str(arq).endswith(".shp"):
            try:
                temp_gdf = gpd.read_file(arq)
                lista_gdfs.append(temp_gdf)
            except Exception as e:
                print(f"⚠️ Erro ao ler {arq}: {e}")
    
    if lista_gdfs:
        gdf_final = pd.concat(lista_gdfs, ignore_index=True)
        # Garante o sistema de coordenadas (WGS84)
        gdf_final.crs = "EPSG:4326" 
        
        # Salva o arquivo final para o QGIS
        gdf_final.to_file("malha_voos_antartica_2009_2018.gpkg", driver="GPKG")
        print(f"\n✅ SUCESSO! Gerado: 'malha_voos_antartica_2009_2018.gpkg'")
        print(f"📊 Total de feições espaciais: {len(gdf_final)}")
    else:
        print("❌ Nenhum arquivo .shp encontrado para consolidar.")

if __name__ == "__main__":
    baixar_e_consolidar_antartica()

📡 Localizando IPFLT1B na Antártica...
📥 Baixando arquivos para ./dados_ipflt1b...


QUEUEING TASKS | :   0%|          | 0/5980 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/5980 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/5980 [00:00<?, ?it/s]

🌍 Consolidando trajetórias em um único GeoPackage...

✅ SUCESSO! Gerado: 'malha_voos_antartica_2009_2018.gpkg'
📊 Total de feições espaciais: 1196
